In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
_r = pathlib.Path.cwd().resolve()
while not (_r / "config" / "settings.yaml").exists() and _r != _r.parent:
    _r = _r.parent
sys.path.insert(0, str(_r))                      # makes `import src...` work from any folder
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
from src.notebook_utils import *
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 40, "display.width", 140)
print("project root:", _r)

project root: /app


# 01 - Data preprocessing
Generate (or load) data → validate → clean → store as Parquet/CSV. Reproducible via seed in `config/settings.yaml`.

In [2]:
from src.data_collection.synthetic import generate_transactions
from src.preprocessing.validation import validate_transactions
from src.preprocessing.cleaning import clean_transactions, frame_hash
s = CFG['synthetic']
raw = generate_transactions(20000, s['n_customers'], s['fraud_rate'], 60, CFG['seed'], '2025-06-30', s['dirty_fraction'])
print(raw.shape)

(20040, 16)


In [3]:
pd.DataFrame(validate_transactions(raw, CFG))[['check','passed','failed_count']]

,check,passed,failed_count
0,schema_columns_present,True,0
1,min_rows,True,0
2,missing_ratio_within_limit,True,0
3,no_missing_amount,False,80
4,duplicate_ratio_within_limit,True,0
5,no_duplicate_transaction_id,False,40
6,amount_positive,False,26
7,timestamp_parsable,True,0
8,ip_risk_in_unit_interval,True,0
9,label_is_binary,True,0


In [4]:
clean, rep = clean_transactions(raw); print(rep)
print('content hash (same seed -> same hash):', frame_hash(clean.drop(columns=['timestamp'])))

{'rows_in': 20040, 'duplicates_removed': 40, 'invalid_removed': 106, 'rows_out': 19894}
content hash (same seed -> same hash): 004d748136600e6f


In [5]:
again, _ = clean_transactions(generate_transactions(20000, s['n_customers'], s['fraud_rate'], 60, CFG['seed'], '2025-06-30', s['dirty_fraction']))
assert frame_hash(again.drop(columns=['timestamp'])) == frame_hash(clean.drop(columns=['timestamp'])); print('reproducible: OK')

reproducible: OK


In [6]:
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    clean.to_parquet(f'{d}/t.parquet', index=False); clean.to_csv(f'{d}/t.csv', index=False)
    print({f: os.path.getsize(f'{d}/{f}')//1024 for f in os.listdir(d)}, 'KiB')

{'t.parquet': 585, 't.csv': 2032} KiB


Parquet keeps dtypes and is several times smaller than CSV; the pipeline stores both (CSV for portability, Parquet for speed).